In [ ]:
import torch
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from PIL import Image
import os
import random
from helper import read_voxel_region

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
#tiff_path = "./data_yukiko/images/default/triax.0.eps=-0.0025.spheres_image_gaussian=08_noise=03.tif"
tiff_path = "./data_yukiko/images/basic/triax.0.eps=-0.0025.spheres_image_gaussian=00_noise=00.tif"

with tifffile.TiffFile(tiff_path) as tif:
    shape = tif.series[0].shape
    print(f"Image dimensions: {shape}")
    print(f"Data type: {tif.series[0].dtype}")
        
    depth, height, width = shape
            
    print(f"Depth (Z): {depth}, Height (Y): {height}, Width (X): {width}")

In [ ]:
REGION_SIZE = 100
RANDOM_OFFSET_RANGE = 50

voxel_region, start_coords = read_voxel_region(tiff_path, region_size=REGION_SIZE, random_offset_range=RANDOM_OFFSET_RANGE)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'{REGION_SIZE}x{REGION_SIZE}x{REGION_SIZE} Voxel Region - Selected Slices', fontsize=16)

slice_indices = [int(REGION_SIZE * ratio) for ratio in [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]]
slice_indices = [min(idx, REGION_SIZE-1) for idx in slice_indices]

for i, slice_idx in enumerate(slice_indices):
    row = i // 3
    col = i % 3
    
    im = axes[row, col].imshow(voxel_region[slice_idx], cmap='gray', interpolation='nearest')
    axes[row, col].set_title(f'Z-slice {slice_idx}')
    axes[row, col].set_xlabel('X')
    axes[row, col].set_ylabel('Y')
    
    plt.colorbar(im, ax=axes[row, col], shrink=0.8)

plt.tight_layout()
plt.show()


## Otsu Segmentation

- https://en.wikipedia.org/wiki/Otsu%27s_method

- https://scikit-image.org/docs/0.23.x/auto_examples/segmentation/plot_thresholding.html

- http://youtube.com/watch?v=YdhhiXDQDl4

In [ ]:
from skimage.filters import threshold_otsu

threshold = threshold_otsu(voxel_region)

print(f"Otsu threshold: {threshold}")

# Use the same dynamic slice indices as above
slice_indices = [int(REGION_SIZE * ratio) for ratio in [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]]
slice_indices = [min(idx, REGION_SIZE-1) for idx in slice_indices]

fig, axes = plt.subplots(3, 6, figsize=(20, 12))
fig.suptitle('Voxel Region Analysis: Original vs Otsu Segmentation', fontsize=16)

for i, slice_idx in enumerate(slice_indices):
    col = i
    
    im_orig = axes[0, col].imshow(voxel_region[slice_idx], cmap='gray', interpolation='nearest')
    axes[0, col].set_title(f'Original Z-slice {slice_idx}')
    axes[0, col].set_xlabel('X')
    axes[0, col].set_ylabel('Y')
    plt.colorbar(im_orig, ax=axes[0, col], shrink=0.6)
    
    binary_mask = voxel_region[slice_idx] > threshold
    
    im_seg = axes[1, col].imshow(binary_mask, cmap='gray', interpolation='nearest')
    axes[1, col].set_title(f'Binary Otsu Z-slice {slice_idx}')
    axes[1, col].set_xlabel('X')
    axes[1, col].set_ylabel('Y')
    
    inverted_mask = ~binary_mask
    axes[2, col].imshow(inverted_mask, cmap='gray', interpolation='nearest')
    axes[2, col].set_title(f'Inverted Mask Z-slice {slice_idx}')
    axes[2, col].set_xlabel('X')
    axes[2, col].set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
import os

def create_triple_comparison_gif(voxel_data, threshold, output_path="triple_segmentation.gif", fps=8):

    depth = voxel_data.shape[0]
    vmin, vmax = np.min(voxel_data), np.max(voxel_data)
    

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Triple View Animation: Original | Binary | Inverted', fontsize=14)
    
    im1 = ax1.imshow(voxel_data[0], cmap='gray', vmin=vmin, vmax=vmax)
    ax1.set_title('Original')
    ax1.axis('off')
    
    im2 = ax2.imshow(voxel_data[0] > threshold, cmap='gray', vmin=0, vmax=1)
    ax2.set_title('Binary Otsu')
    ax2.axis('off')
    
    im3 = ax3.imshow(~(voxel_data[0] > threshold), cmap='gray', vmin=0, vmax=1)
    ax3.set_title('Inverted')
    ax3.axis('off')
    
    progress_text = fig.text(0.5, 0.02, '', ha='center', fontsize=10)
    
    def animate(frame):
        im1.set_array(voxel_data[frame])
        im1.set_clim(vmin, vmax)
        
        binary_mask = voxel_data[frame] > threshold
        im2.set_array(binary_mask)
        
        inverted_mask = ~binary_mask
        im3.set_array(inverted_mask)
        
        foreground_pct = (np.sum(binary_mask) / binary_mask.size) * 100
        progress = (frame + 1) / depth * 100
        progress_text.set_text(f'Progress: {progress:.1f}% | Slice {frame} | Foreground: {foreground_pct:.1f}%')
        
        return [im1, im2, im3, progress_text]
    
    anim = animation.FuncAnimation(fig, animate, frames=depth, interval=1000//fps, blit=False, repeat=True)
    
    print(f"Creating triple-view GIF with {depth} frames at {fps} FPS...")
    writer = PillowWriter(fps=fps)
    anim.save(output_path, writer=writer)
    
    plt.close(fig)
    return anim


anim = create_triple_comparison_gif(voxel_region, threshold, "images/triple_segmentation.gif", fps=8)

if os.path.exists("images/triple_segmentation.gif"):
    size_mb = os.path.getsize("images/triple_segmentation.gif") / (1024 * 1024)
else:
    print("Error: GIF file was not created successfully.")

## 3D Interactive Visualization

Creating an interactive 3D visualization of the segmented sand grains using Plotly

In [ ]:
from skimage import measure
import numpy as np
import time

def create_interactive_3d_visualization(voxel_data, threshold, output_path="3d_interactive.html"):
    try:
        import plotly.graph_objects as go
        import plotly.offline as pyo
        
        start_time = time.time()
        
        binary_volume = voxel_data > threshold
        
        if np.sum(binary_volume) == 0:
            print("Error: No foreground data found after thresholding!")
            return None
        
        volume_size = binary_volume.shape[0]
        print(f"Processing full resolution volume: {binary_volume.shape}")
        
        vertices, faces, normals, values = measure.marching_cubes(binary_volume, level=0.5)
        
        print(f"Generated {len(vertices)} vertices and {len(faces)} faces")
        
        fig = go.Figure(data=[
            go.Mesh3d(
                x=vertices[:, 2],
                y=vertices[:, 1], 
                z=vertices[:, 0],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color='lightblue',
                opacity=0.8,
                lighting=dict(ambient=0.18, diffuse=1, fresnel=0.1, specular=1, roughness=0.1),
                lightposition=dict(x=100, y=200, z=0),
                name="Sand Grain 3D Model",
                flatshading=False,
                alphahull=0,
                delaunayaxis='z',
                showscale=False,
                hoverinfo='none'
            )
        ])
        
        fig.update_layout(
            title=f"Interactive 3D Sand Grain Segmentation ({volume_size}³ voxels)",
            scene=dict(
                xaxis_title="X (voxels)",
                yaxis_title="Y (voxels)", 
                zaxis_title="Z (voxels)",
                bgcolor="black",
                camera=dict(eye=dict(x=1.2, y=1.2, z=0.6)),
                aspectmode='cube'
            ),
            paper_bgcolor="black",
            plot_bgcolor="black",
            font=dict(color="white"),
            width=None, 
            height=None, 
            autosize=True,
            margin=dict(l=0, r=0, t=50, b=0)
        )
        
        config = {
            'displayModeBar': True,
            'displaylogo': False,
            'modeBarButtonsToRemove': ['pan2d', 'lasso2d'],
            'responsive': True
        }
        
        pyo.plot(fig, filename=output_path, auto_open=False, config=config)
        
        try:
            with open(output_path, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            full_screen_css = """
            <style>
            html, body {
                margin: 0;
                padding: 0;
                height: 100vh;
                width: 100vw;
                overflow: hidden;
                background: black;
            }
            #plotly-div {
                width: 100vw;
                height: 100vh ;
                margin: 0 ;
                padding: 0 ;
            }
            .plotly-graph-div {
                width: 100% ;
                height: 100% ;
            }
            </style>
            """
            
            html_content = html_content.replace('</head>', full_screen_css + '</head>')
            
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(html_content)
                
        except Exception as e:
            print(f"Could not modify HTML for full-screen (still works): {e}")
        
        return output_path
        
    except ImportError:
        print("Plotly not installed. Install with: pip install plotly")
        print("This is required for interactive 3D visualization.")
        return None
    except Exception as e:
        print(f"Interactive visualization failed: {e}")
        return None

In [ ]:
result = create_interactive_3d_visualization(
    voxel_region, 
    threshold, 
    "3d_interactive.html"
)

if result:
    if os.path.exists(result):
        size_mb = os.path.getsize(result) / (1024 * 1024)
else:
    print("Failed to create interactive visualization")